# Parking Lot Vehicle Monitoring with YOLO

**Goal of this notebook**

Build an end-to-end pipeline, built for Kaggle Notebooks, that uses the GPU by default (GPU `0` — the first of the T4 x2) and can be switched to CPU with a single boolean flag (Section 3), that:

1. Trains a YOLO object detector on a vehicle-image dataset (cars + motorcycles).
2. Uses the trained detector on periodic photos of a small parking lot to:
   - Detect whether **specific parking spots** (calibrated once, by hand) are occupied or empty.
   - Track vehicles across photos to determine when a car **enters**, **leaves**, and **how long it stayed**.
3. As a final, separate module: given a single parking-lot photo, outputs for each detected vehicle its **dominant color**, and for cars specifically, a best-guess **brand and model** (zero-shot, no extra training required).

**Design decisions locked in from our discussion (so future-you remembers why):**

- Training uses a **generic, already-YOLO-formatted vehicle dataset** (Roboflow's `Vehicles-OpenImages`, classes: `Car`, `Motorcycle`, `Bus`, `Truck`, `Ambulance`). It is not parking-lot-specific, so accuracy on your exact camera angle may need a later fine-tuning pass with your own labeled photos.
- Everything runs on the **GPU by default** (GPU `0` on Kaggle's T4 x2). One boolean flag (Section 3) switches the entire notebook to CPU — nothing else changes.
- Parking spots are defined as **polygons, not axis-aligned boxes**, because a wide/fisheye camera angle distorts spot shapes into trapezoids (see calibration section).
- The **walkway area is intentionally excluded** from the spot list — it is not a real parking spot, just floor space.
- The **motorcycle spot only checks for presence** of a motorcycle (no brand/model needed there) — color is a nice-to-have.
- **No OCR / timestamp reading is baked into training or detection.** The burned-in camera timestamp is never read or used to alter training images in any way. For the entry/exit/duration tracking demo, timestamps come from an external list you provide (e.g. derived from filenames) — fully decoupled from the model pipeline.

Let's get started.

## 1. Install dependencies

We need:
- `ultralytics` — YOLO training/inference (YOLOv8).
- `roboflow` — to download the training dataset in YOLO format.
- `shapely` — polygon geometry for parking-spot occupancy checks.
- `opencv-python-headless` — image loading/drawing.
- `scikit-learn` — K-Means for dominant-color extraction.
- `open_clip_torch` — pretrained CLIP model, used **zero-shot** (no training) for brand/model guessing in the final module.
- `pandas` — building the entry/exit/duration log as a table.

Colab already has `torch`, `numpy`, `matplotlib`, and `Pillow` preinstalled, so we don't reinstall those.

In [ ]:
!pip install -q ultralytics roboflow shapely opencv-python-headless scikit-learn open_clip_torch pandas

## 2. Imports

Standard imports for the whole notebook. Nothing here trains or downloads anything yet.

In [ ]:
import os
import json
import glob
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from shapely.geometry import Polygon, box as shapely_box

import torch
from ultralytics import YOLO

print("Setup OK. Libraries imported.")

## 3. Choose the device (GPU / CPU)

One boolean controls the device for the **entire** notebook — every training and inference cell below reads `FORCE_DEVICE` from this cell:

- **`USE_CPU = False` (default)** → uses the GPU, device `0` (the first GPU of Kaggle's T4 x2).
- **`USE_CPU = True`** → forces `"cpu"` everywhere.

Flip that one flag and nothing else in the notebook needs to change.

In [ ]:
# Single switch for the whole notebook:
#   False -> GPU (device 0 — the first GPU of Kaggle's T4 x2)
#   True  -> CPU
USE_CPU = False

if USE_CPU:
    FORCE_DEVICE = "cpu"
else:
    FORCE_DEVICE = 0  # first GPU (T4 x2 on Kaggle)

# Model selection — change this single variable to switch models
MODEL_NAME = "yolov8m.pt"  # e.g. "yolov8n.pt", "yolov8s.pt", "yolov8m.pt", "yolov11n.pt", etc.

print(f"FORCE_DEVICE set to: {FORCE_DEVICE!r}")
print(f"MODEL_NAME: {MODEL_NAME}")
if not USE_CPU and torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  - cuda:{i} -> {torch.cuda.get_device_name(i)}")

## 4. Get the training dataset

We use **`Vehicles-OpenImages`**, a public dataset on Roboflow Universe with the classes: `Ambulance`, `Bus`, `Car`, `Motorcycle`, `Truck`. This gives us both `Car` and `Motorcycle` — the two classes our parking lot needs.

Two versions are available as Kaggle inputs:

- **`vehicles-openimages`** — standard YOLO format (xywh labels), used by the default `yolov8` variant
- **`vehicles-openimages-obb`** — OBB format (4-corner rotated labels), used by the `yolo26_obb` variant

The active dataset is selected by `MODEL_VARIANT` in Section 3.

**Make sure you've added the zip as a Kaggle Input dataset (see the steps above) before running this.**

In [ ]:
# Dataset path (explicit — no glob searching)
DATASET_DIR = "/kaggle/input/datasets/tomasreisenman/vehicles-openimages"
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

print(f"Using dataset: {DATASET_DIR}")
print(f"Using model: {MODEL_NAME}")

## 5. Inspect the dataset structure

Before training, let's sanity-check what we actually downloaded: the `data.yaml` (class names + folder layout) and how many images/labels exist per split.

In [ ]:
# data_yaml_path is already set in Section 4
with open(data_yaml_path, "r") as f:
    print(f.read())

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    if os.path.isdir(img_dir):
        n_images = len(glob.glob(os.path.join(img_dir, "*")))
        print(f"{split}: {n_images} images")
    else:
        print(f"{split}: folder not found")

## 6. Visual sanity check of the training data

Let's plot a handful of training images with their ground-truth bounding boxes drawn, just to confirm the labels look correct before we spend CPU time training on them.

In [ ]:
def yolo_txt_to_boxes(label_path, img_w, img_h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = map(float, parts)
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append((int(cls), x1, y1, x2, y2))
    return boxes


train_img_dir = os.path.join(DATASET_DIR, "train", "images")
train_lbl_dir = os.path.join(DATASET_DIR, "train", "labels")
sample_imgs = glob.glob(os.path.join(train_img_dir, "*"))[:4]

fig, axes = plt.subplots(1, len(sample_imgs), figsize=(20, 5))
for ax, img_path in zip(axes, sample_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label_path = os.path.join(train_lbl_dir, Path(img_path).stem + ".txt")
    for cls, x1, y1, x2, y2 in yolo_txt_to_boxes(label_path, w, h):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Train YOLO

Training choices, and why — tuned for the default GPU runtime (T4 x2); flip the boolean in Section 3 to switch to CPU:

- **Model: `yolov8m.pt`** (medium) — current choice, see the comparison in Section 7b. It won on final validation mAP50 among the models tested. On CPU it is noticeably slower per epoch than nano/small, but on GPU (like your T4 x2) it trains in minutes — change `MODEL_NAME` in Section 3 to try another variant.
- **Image size: 416** instead of the default 640 — smaller images mean roughly 2-3x faster CPU inference/training with a moderate accuracy trade-off. On GPU this constraint matters less, but it's left as-is to keep results comparable across runs.
- **Epochs: 30** — enough to see the loss curves converge without an excessive run time on GPU.
- **Batch size: auto on GPU, 8 on CPU** — `batch=-1` tells `ultralytics` to automatically pick the largest batch size that fits in GPU memory, which uses your T4s far more efficiently than a fixed small batch. On CPU there's no such benefit, so it stays fixed at 8 to keep memory use predictable.

On CPU, expect this to take a while (potentially 30+ minutes). On GPU it should be a few minutes. Ultralytics prints per-epoch progress so you can monitor it either way.

In [ ]:
model = YOLO(MODEL_NAME)  # starts from COCO-pretrained weights, then fine-tunes on our dataset

train_batch = -1 if FORCE_DEVICE != "cpu" else 8  # auto-batch on GPU, fixed small batch on CPU

results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=416,
    batch=train_batch,
    device=FORCE_DEVICE,   # from Section 3 — 0 (first GPU) by default, "cpu" if USE_CPU
    workers=2,
    # --- Data augmentation ---
    hsv_h=0.015,      # slight hue jitter
    hsv_s=0.7,        # saturation jitter — helps with lighting/paint-color variety
    hsv_v=0.4,        # brightness jitter — useful given your day/night frames
    degrees=15.0,     # moderate rotation
    translate=0.1,    # small positional jitter
    scale=0.5,        # scale jitter — helps with camera-distance/zoom mismatch
    shear=2.0,        # mild shear — approximates minor perspective variation
    perspective=0.0002,
    flipud=0.0,       # vertical flip — disabled: camera is oblique, not top-down
    fliplr=0.5,       # horizontal flip, standard
    mosaic=1.0,       # combine 4 images per batch
    mixup=0.1,        # light regularization
    erasing=0.4,      # random erasing
    project="parking_yolo",
    name="train_run",
    patience=10,           # stop early if validation stops improving
)

BEST_WEIGHTS = os.path.join(results.save_dir, "weights", "best.pt")
print(f"Best weights saved at: {BEST_WEIGHTS}")

## 7b. Training results comparison — YOLOv8 / YOLO11 / YOLO26 (n, s, m)

Snapshot of the runs logged in `../results.md` (all at `imgsz=416` on the Vehicles-OpenImages validation split: 250 images, 454 instances).

**Winner: `yolov8m` ✅** — highest validation mAP50, hence `MODEL_NAME = "yolov8m.pt"` in Section 3.

### What the metrics mean

- **mAP50** — mean Average Precision at IoU ≥ 0.5, averaged over all classes. The headline detection-quality number: higher is better.
- **mAP50-95** — same, but averaged over IoU thresholds 0.5–0.95. Stricter; rewards tight boxes, not just roughly-correct ones.
- **P (precision)** — of all boxes the model predicted, the fraction that was correct. Low precision = many false alarms.
- **R (recall)** — of all real objects, the fraction the model found. Low recall = many missed cars.
- **box_loss** — error in predicted box coordinates. Lower is better, but only comparable between runs of the same setup.
- **cls_loss** — error in predicted class labels (Car vs Bus vs ...). Lower is better.
- **dfl_loss** — distribution focal loss, fine-grained quality of box edges. Lower is better.
- **Params / GFLOPs** — model size and compute cost. Bigger models score higher but train/run slower.

### Results

| Model | Params | GFLOPs | box_loss | cls_loss | dfl_loss | P | R | mAP50 | mAP50-95 |
|---|---|---|---|---|---|---|---|---|---|
| yolov8n | 3.2M | 8.7 | 0.695 | 0.690 | 1.030 | 0.595 | 0.552 | 0.582 | 0.457 |
| yolov8s | 11.2M | 28.6 | 0.750 | 0.680 | 1.100 | 0.610 | 0.580 | 0.595 | 0.440 |
| **yolov8m ✅** | **25.8M** | **78.7** | 0.809 | 0.613 | 1.171 | 0.650 | 0.563 | **0.607** | 0.430 |
| yolo11n | 2.6M | 6.5 | 0.604 | 0.448 | 0.981 | 0.590 | 0.642 | 0.598 | 0.460 |
| yolo11s | 9.4M | 21.5 | 0.980 | 0.990 | 1.300 | 0.600 | 0.590 | 0.586 | 0.405 |
| yolo11m | 20.0M | 67.8 | 0.947 | 0.926 | 1.270 | 0.575 | 0.600 | 0.581 | 0.412 |
| yolo26n | 2.9M | 7.2 | 0.660 | 0.550 | 1.020 | 0.570 | 0.600 | 0.570 | 0.420 |
| yolo26s | 10.5M | 25.0 | 0.800 | 0.700 | 1.150 | 0.600 | 0.590 | 0.590 | 0.425 |
| yolo26m | 22.0M | 72.0 | 0.850 | 0.680 | 1.190 | 0.630 | 0.580 | 0.602 | 0.428 |

## 8. Validate the trained model

Run the built-in validation to get standard detection metrics (precision, recall, mAP@0.5, mAP@0.5:0.95) on the held-out validation split. These numbers tell us how much to trust the detector before relying on it for occupancy/tracking logic.

In [ ]:
trained_model = YOLO(BEST_WEIGHTS)
metrics = trained_model.val(data=data_yaml_path, imgsz=416, device=FORCE_DEVICE)

print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

## 9. Quick inference sanity check

Before moving to parking-lot-specific logic, let's confirm the trained model draws sensible boxes on a few validation images.

In [ ]:
val_img_dir = os.path.join(DATASET_DIR, "valid", "images")
test_samples = glob.glob(os.path.join(val_img_dir, "*"))[:4]

pred_results = trained_model.predict(test_samples, imgsz=416, device=FORCE_DEVICE, conf=0.25)

fig, axes = plt.subplots(1, len(pred_results), figsize=(20, 5))
for ax, r in zip(axes, pred_results):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9b. Rotation alignment check — vertical vs horizontal vs original

The longest structural line in the reference photo (spot edge, curb) sets the
parking-lot direction. From its angle we build **two** rotations:

- **vertical (current)** — rotates by `90 - angle`, so the longest line ends up vertical.
- **horizontal (previous)** — rotates by `-angle`, so the longest line ends up horizontal.
- **original** — no rotation at all.

All three variants of every photo in `reference/` are run through the trained
model, so we can see which orientation YOLO detects best.

In [ ]:
# --- 9b setup: line detection + both rotation matrices ---
# Reference photos live here (same folder Section 12 uses later as BASE_DIR)
REF_DIR = "/kaggle/input/datasets/tomasreisenman/reference/reference/"
REFERENCE_IMAGE_FOR_HOMOGRAPHY = REF_DIR + "reference_image.jpg"

import glob as _glob
REF_IMAGES = sorted(_glob.glob(os.path.join(REF_DIR, "*.jpg")))
print(f"Found {len(REF_IMAGES)} reference photos in {REF_DIR}")
for p in REF_IMAGES:
    print(f"  {os.path.basename(p)}")


def preprocess_for_lines(image_bgr, blur_kernel=(5, 5)):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.GaussianBlur(gray, blur_kernel, 0)


def line_angle_deg(x1, y1, x2, y2):
    return np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180  # direction only, 0-180


def filter_border_lines(lines, image_shape, border_threshold=10, angle_threshold=10):
    """Discard lines nearly parallel to an image border AND close to it.

    The Hough transform often detects the image border itself as a line,
    which is not a useful structural line.
    """
    if len(lines) == 0:
        return lines
    h, w = image_shape[:2]
    filtered = []
    for line in lines:
        x1, y1, x2, y2 = line
        angle = line_angle_deg(x1, y1, x2, y2)
        is_horizontal = angle < angle_threshold or angle > (180 - angle_threshold)
        if is_horizontal:
            if min(y1, y2) < border_threshold or max(y1, y2) > (h - border_threshold):
                continue
        is_vertical = (90 - angle_threshold) < angle < (90 + angle_threshold)
        if is_vertical:
            if min(x1, x2) < border_threshold or max(x1, x2) > (w - border_threshold):
                continue
        filtered.append(line)
    return np.array(filtered, dtype=int) if filtered else np.empty((0, 4), dtype=int)


def detect_line_segments(image_bgr, canny_low=50, canny_high=150,
                          hough_threshold=60, min_line_length=60, max_line_gap=15):
    gray = preprocess_for_lines(image_bgr)
    edges = cv2.Canny(gray, canny_low, canny_high)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=hough_threshold,
                             minLineLength=min_line_length, maxLineGap=max_line_gap)
    if lines is None:
        return np.empty((0, 4), dtype=int)
    lines = lines.reshape(-1, 4)
    return filter_border_lines(lines, image_bgr.shape, border_threshold=10, angle_threshold=10)


def line_length(line):
    x1, y1, x2, y2 = line
    return np.hypot(x2 - x1, y2 - y1)


def longest_line(lines):
    """Return the single longest line from the array."""
    if len(lines) == 0:
        return None
    lengths = np.array([line_length(l) for l in lines])
    return lines[np.argmax(lengths)]


def rotation_matrices_for_angle(angle_deg, image_shape):
    """Build both rotations from the longest-line angle.

    Returns (M_vertical, M_horizontal): 2x3 matrices for cv2.warpAffine.
    """
    h, w = image_shape[:2]
    center = (w // 2, h // 2)
    M_vertical = cv2.getRotationMatrix2D(center, 90 - angle_deg, 1.0)    # current
    M_horizontal = cv2.getRotationMatrix2D(center, -angle_deg, 1.0)     # previous
    return M_vertical, M_horizontal


def apply_rotation(image_bgr, M):
    h, w = image_bgr.shape[:2]
    return cv2.warpAffine(image_bgr, M, (w, h), borderMode=cv2.BORDER_REPLICATE)


# --- Calibrate once on the clearest reference photo ---
ref_img = cv2.imread(REFERENCE_IMAGE_FOR_HOMOGRAPHY)
if ref_img is None:
    raise FileNotFoundError(f"Could not read {REFERENCE_IMAGE_FOR_HOMOGRAPHY}")

detected_lines = detect_line_segments(ref_img)
LONGEST = longest_line(detected_lines)
if LONGEST is None:
    raise RuntimeError("No lines detected in the reference image — try adjusting Hough params.")

LONGEST_ANGLE = line_angle_deg(*LONGEST)
if LONGEST_ANGLE >= 90:
    LONGEST_ANGLE -= 180
print(f"Longest line: {LONGEST.tolist()}  angle: {LONGEST_ANGLE:.1f} deg")

ROTATION_VERTICAL, ROTATION_HORIZONTAL = rotation_matrices_for_angle(LONGEST_ANGLE, ref_img.shape)

# Debug view: what the line detector sees
preprocessed = preprocess_for_lines(ref_img)
edges_vis = cv2.Canny(preprocessed, 50, 150)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Reference (original)")
axes[0].axis("off")
axes[1].imshow(preprocessed, cmap="gray")
axes[1].set_title("Blurred greyscale (line-detection input)")
axes[1].axis("off")
axes[2].imshow(edges_vis, cmap="gray")
axes[2].set_title("Canny edges")
axes[2].axis("off")
plt.tight_layout()
plt.show()

# Longest line overlay + both rotations of the reference
lines_vis = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB).copy()
x1, y1, x2, y2 = [int(v) for v in LONGEST]
cv2.line(lines_vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].imshow(lines_vis)
axes[0].set_title(f"Longest line (red, {LONGEST_ANGLE:.1f} deg)")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(apply_rotation(ref_img, ROTATION_VERTICAL), cv2.COLOR_BGR2RGB))
axes[1].set_title("Vertical rotation (current)")
axes[1].axis("off")
axes[2].imshow(cv2.cvtColor(apply_rotation(ref_img, ROTATION_HORIZONTAL), cv2.COLOR_BGR2RGB))
axes[2].set_title("Horizontal rotation (previous)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- 9b comparison: original vs vertical vs horizontal on all reference photos ---
def predict_boxes(image_bgr, model, imgsz=416, conf=0.25):
    res = model.predict(image_bgr, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]
    boxes = []
    if res.boxes is not None:
        for b in res.boxes:
            boxes.append((res.names[int(b.cls)], float(b.conf), [round(v, 1) for v in b.xyxy[0].tolist()]))
    return res, boxes


combo_rows = []
for img_path in REF_IMAGES:
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f"Skipping unreadable: {img_path}")
        continue
    variants = {
        "original": img_bgr,
        "vertical": apply_rotation(img_bgr, ROTATION_VERTICAL),
        "horizontal": apply_rotation(img_bgr, ROTATION_HORIZONTAL),
    }
    print(f"\n=== {os.path.basename(img_path)} ===")
    plotted = {}
    for name, variant in variants.items():
        res, boxes = predict_boxes(variant, trained_model)
        confs = [c for _, c, _ in boxes]
        combo_rows.append({
            "image": os.path.basename(img_path),
            "variant": name,
            "n_detections": len(boxes),
            "mean_conf": round(float(np.mean(confs)), 3) if confs else 0.0,
        })
        print(f"  {name}: {len(boxes)} detections" + (f"  mean_conf={np.mean(confs):.3f}" if confs else ""))
        for cls_name, cf, bx in boxes:
            print(f"    {cls_name}  conf={cf:.3f}  box={bx}")
        plotted[name] = res.plot()

    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    for ax, name in zip(axes, ["original", "vertical", "horizontal"]):
        ax.imshow(cv2.cvtColor(plotted[name], cv2.COLOR_BGR2RGB))
        ax.set_title(name)
        ax.axis("off")
    plt.suptitle(os.path.basename(img_path))
    plt.tight_layout()
    plt.show()

combo_df = pd.DataFrame(combo_rows)
print(combo_df.pivot_table(index="image", columns="variant", values="n_detections", aggfunc="first"))
print(combo_df.pivot_table(index="image", columns="variant", values="mean_conf", aggfunc="first"))

## 10. Auto-calibrate parking spots from a reference image

Instead of hard-coding polygon coordinates, this section uses a **reference image with cars already parked** and the trained model to automatically extract the parking spots.

**How it works:**
1. Load the reference image (cars parked in their spots)
2. Run the trained model to detect all vehicles
3. For each detected vehicle, create a parking spot polygon around it (bounding box expanded by `SPOT_EXPANSION` to approximate the real spot boundaries)
4. Visualize the extracted spots for verification

**One-time per camera position** — if the camera moves, re-run this section with a new reference image.

In [ ]:
# --- Auto-calibrate parking spots from a reference image with cars ---
CALIBRATION_IMAGE_PATH = "/kaggle/input/datasets/tomasreisenman/reference/reference/20260912171801.jpg"

# Expansion factor: cars are smaller than spots, so we expand each detection
# bounding box by this fraction on each side to approximate the spot boundaries.
# Increase if spots are too tight, decrease if they overlap too much.
SPOT_EXPANSION = 0.3

# Load calibration image and run detection
calib_img_bgr = cv2.imread(CALIBRATION_IMAGE_PATH)
if calib_img_bgr is None:
    raise FileNotFoundError(f"Could not read {CALIBRATION_IMAGE_PATH}")

calib_result = trained_model.predict(CALIBRATION_IMAGE_PATH, imgsz=416, device=FORCE_DEVICE, conf=0.25, verbose=False)[0]

# Extract parking spots from detections
PARKING_SPOTS = {}
spot_idx = 1
for b in calib_result.boxes:
    cls_name = calib_result.names[int(b.cls)]
    if cls_name not in ("Car", "Motorcycle"):
        continue
    x1, y1, x2, y2 = b.xyxy[0].tolist()
    w = x2 - x1
    h = y2 - y1
    # Expand bounding box to approximate spot size
    x1_exp = max(0, int(x1 - w * SPOT_EXPANSION))
    y1_exp = max(0, int(y1 - h * SPOT_EXPANSION))
    x2_exp = min(calib_img_bgr.shape[1], int(x2 + w * SPOT_EXPANSION))
    y2_exp = min(calib_img_bgr.shape[0], int(y2 + h * SPOT_EXPANSION))
    # Create polygon (rectangle in image coordinates)
    polygon = [(x1_exp, y1_exp), (x2_exp, y1_exp), (x2_exp, y2_exp), (x1_exp, y2_exp)]
    vehicle_type = "motorcycle" if cls_name == "Motorcycle" else "car"
    PARKING_SPOTS[f"spot_{spot_idx}_{vehicle_type}"] = {
        "polygon": polygon,
        "vehicle_type": vehicle_type
    }
    spot_idx += 1

print(f"Extracted {len(PARKING_SPOTS)} parking spots from calibration image:")
for name, spot in PARKING_SPOTS.items():
    print(f"  {name}: {spot['vehicle_type']}  polygon={spot['polygon']}")

# Visualize the extracted spots
def draw_calibration(image_path, spots):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    overlay = img.copy()
    for name, spot in spots.items():
        pts = np.array(spot["polygon"], dtype=np.int32)
        color = (0, 255, 255) if spot["vehicle_type"] == "car" else (255, 0, 255)
        cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=3)
        cx, cy = pts.mean(axis=0).astype(int)
        cv2.putText(overlay, name, (cx - 40, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(10, 8))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title("Auto-extracted parking spots — do these align with the real spots?")
    plt.show()

draw_calibration(CALIBRATION_IMAGE_PATH, PARKING_SPOTS)

## 11. Per-spot occupancy detection

For each calibrated spot, we check whether any detected vehicle box of the **matching class** overlaps it enough to count as "occupied":

- `vehicle_type == "car"` → only `Car` detections are considered.
- `vehicle_type == "motorcycle"` → only `Motorcycle` detections are considered (as agreed, this spot never checks for cars, brand, or model).

Overlap is measured as **intersection area / spot polygon area** using `shapely` (not a plain box IoU, since spots are irregular quadrilaterals). A spot is "occupied" if this ratio passes a threshold (default 30% — tune this once you see it running on real photos).

In [ ]:
OCCUPANCY_THRESHOLD = 0.80

def check_occupancy(image_path, model, spots, conf=0.25, imgsz=416):
    result = model.predict(image_path, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]
    names = result.names  # class id -> class name

    detections = []
    for b in result.boxes:
        cls_name = names[int(b.cls)]
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        detections.append({"class": cls_name, "box": (x1, y1, x2, y2), "conf": float(b.conf)})

    occupancy = {}
    for spot_name, spot in spots.items():
        spot_poly = Polygon(spot["polygon"])
        expected_class = "Car" if spot["vehicle_type"] == "car" else "Motorcycle"

        occupied = False
        matched_box = None
        for det in detections:
            if det["class"] != expected_class:
                continue
            det_poly = shapely_box(*det["box"])
            if not spot_poly.intersects(det_poly):
                continue
            overlap_ratio = spot_poly.intersection(det_poly).area / spot_poly.area
            if overlap_ratio >= OCCUPANCY_THRESHOLD:
                occupied = True
                matched_box = det["box"]
                break

        occupancy[spot_name] = {"occupied": occupied, "box": matched_box}

    return occupancy, detections

## 12. Run occupancy check on a single frame and visualize it

Green outline = occupied, red outline = empty — a quick visual confirmation that matches what a human glancing at the photo would say.

The cell also verifies that `TEST_FRAME_PATH` and every image in `TEST_IMAGES` (used by the two debug cells below) actually exists under `BASE_DIR`.

In [ ]:
def visualize_occupancy(image_path, occupancy, spots):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    for spot_name, spot in spots.items():
        pts = np.array(spot["polygon"], dtype=np.int32)
        is_occupied = occupancy[spot_name]["occupied"]
        color = (0, 200, 0) if is_occupied else (220, 0, 0)
        cv2.polylines(img, [pts], isClosed=True, color=color, thickness=3)
        label = f"{spot_name}: {'OCCUPIED' if is_occupied else 'EMPTY'}"
        cx, cy = pts.mean(axis=0).astype(int)
        cv2.putText(img, label, (cx - 60, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

BASE_DIR = "/kaggle/input/datasets/tomasreisenman/reference/reference/"

TEST_FRAME_PATH = BASE_DIR + "20260915212238" + ".jpg"

TEST_IMAGES = [
    BASE_DIR + "20260910200600.jpg",
    BASE_DIR + "20260911172532.jpg",
    BASE_DIR + "20260911194005.jpg",
    BASE_DIR + "20260912171801.jpg",
    BASE_DIR + "20260912191134.jpg",
    BASE_DIR + "20260913005125.jpg",
    BASE_DIR + "20260914191018.jpg",
    BASE_DIR + "20260915212238.jpg",
    BASE_DIR + "20260916110642.jpg",
    BASE_DIR + "20260920190053.jpg",
    BASE_DIR + "reference_image.jpg",
]

missing = [p for p in [TEST_FRAME_PATH] + TEST_IMAGES if not os.path.exists(p)]
if missing:
    print("Warning — not found (check the dataset was added and BASE_DIR is correct):")
    for p in missing:
        print(f"  {p}")
else:
    print(f"All {len(TEST_IMAGES)} test images found.")

if os.path.exists(TEST_FRAME_PATH):
    occupancy, detections = check_occupancy(TEST_FRAME_PATH, trained_model, PARKING_SPOTS)
    print(occupancy)
    visualize_occupancy(TEST_FRAME_PATH, occupancy, PARKING_SPOTS)
else:
    print("TEST_FRAME_PATH not found — skipping the occupancy check.")

In [ ]:
# See every raw detection with its confidence, before spot-matching logic touches it
raw_results = {}
for img_path in TEST_IMAGES:
    if not os.path.exists(img_path):
        print(f"Skipping missing: {img_path}")
        continue
    result = trained_model.predict(img_path, imgsz=416, device=FORCE_DEVICE, conf=0.25, verbose=False)[0]
    raw_results[img_path] = result
    print(f"\n{os.path.basename(img_path)}")
    for b in result.boxes:
        box = [round(v, 1) for v in b.xyxy[0].tolist()]
        print(f"  {result.names[int(b.cls)]}  conf={float(b.conf):.3f}  box={box}")

In [ ]:
# Visualize the raw boxes directly on the image
import matplotlib.pyplot as plt

paths = [p for p in TEST_IMAGES if p in raw_results]
cols = 4
rows = (len(paths) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
for ax, img_path in zip(axes.flat, paths):
    ax.imshow(cv2.cvtColor(raw_results[img_path].plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis("off")
for ax in axes.flat[len(paths):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
import glob

counts = Counter()
for label_file in glob.glob(f"{DATASET_DIR}/train/labels/*.txt"):
    with open(label_file) as f:
        for line in f:
            cls_id = int(line.split()[0])
            counts[cls_id] += 1

print(counts)  # compare count for class id 3 (Motorcycle) vs class id 2 (Car)